# RAG for Code Generation

Retrieval-Augmented Generation (RAG) combines a **retriever** with a **generator** to produce grounded outputs. Instead of relying only on what the model memorized during training, we give it relevant examples at inference time.

![RAG Pipeline](https://docs.aws.amazon.com/images/sagemaker/latest/dg/images/jumpstart/jumpstart-fm-rag.jpg)

Three components:

| Component | What it does | Our choice |
|---|---|---|
| **Knowledge Base** | Stores code examples as vectors in a searchable index | CodeXGLUE (Java + Python) + FAISS |
| **Retriever** | Finds similar examples via cosine similarity | Word2Vec embeddings trained on code |
| **Generator** | Produces code using retrieved examples as few-shot context | Qwen2.5-Coder |

**FAISS** (Facebook AI Similarity Search) is an open-source library by Meta for efficient similarity search over dense vectors. Given a query vector, it finds the nearest neighbors in the index using distance metrics like L2 (Euclidean). We use `IndexFlatL2` — exact brute-force search, simple and accurate for our knowledge base size.

Key idea: the few-shot examples in the prompt are **not random** — they are selected by the retriever based on semantic similarity to the input query.

## 1. Imports and Configuration

In [2]:
import os
import re
import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
from datasets import DatasetDict
import faiss
from tqdm import tqdm
from gensim.models import Word2Vec
from transformers import AutoTokenizer, T5ForConditionalGeneration, AutoModelForCausalLM, AutoTokenizer
import torch

# paths
JAVA_DATASET_PATH = "/home/mhaque/QLoRA-Code-Summarization/Multitask/code-generation/codegen_codexglue/java"
PYTHON_DATASET_PATH = "/home/mhaque/QLoRA-Code-Summarization/Multitask/code-generation/codegen_codexglue/python"
OUTPUT_DIR = "/scratch/mhaque/RAG-Tutorials/dataset/rag_codegen_kb"

print("imports done")

/home/mhaque/.conda/envs/multitask2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mhaque/.conda/envs/multitask2/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


imports done


## 2. Code Embedding Model (Word2Vec Skip-gram)

To retrieve similar code examples, we need to represent each code snippet as a fixed-size vector. We train **Word2Vec** on our code corpus using **skip-gram** mode.

**How skip-gram works:** Given a target token, the model learns to predict nearby tokens. "Nearby" is defined by a **window** — how many tokens to the left and right the model looks.
```
Code:    "public  void  sort  int  arr"
              ↑      ↑    ↑     ↑    ↑
Position:     1      2    3     4    5

With window = 2, for target "sort" (position 3):
  → look 2 left:  "public", "void"
  → look 2 right: "int", "arr"
  → model learns: "sort" is related to all four

With window = 2, for target "int" (position 4):
  → look 2 left:  "void", "sort"
  → look 2 right: "arr"
  → model learns: "int" is related to these three
```

Over millions of code snippets, tokens that keep appearing near similar neighbors (like `sort` and `reverse` both appearing near `array`, `list`, `int[]`) end up with similar vectors.

**Why skip-gram over CBOW?**

| | CBOW | Skip-gram |
|---|---|---|
| **Direction** | Context → predict target | Target → predict context |
| **Strength** | Fast, good for frequent tokens | Better for rare tokens |
| **For code** | Weaker — many rare identifiers in code | Better — captures rare method names and variables |

**Token → Sequence embedding:** Word2Vec gives one vector per token, but FAISS needs one vector per code snippet. We use **mean pooling** — average all token vectors in a snippet into a single vector.
```
"sort an array of integers"
    ↓ tokenize
["sort", "an", "array", "of", "integers"]
    ↓ Word2Vec lookup
[vec_sort, vec_an, vec_array, vec_of, vec_integers]
    ↓ mean pool
[single 128-dim vector]  →  this goes into FAISS
```

In [3]:
def tokenize_code(text):
    """simple whitespace tokenizer, lowercase"""
    return text.lower().split()

class CodeEmbedder:
    """train word2vec on code, then mean-pool token vectors to get sequence embeddings"""

    def __init__(self, vector_size=128, window=5, min_count=2, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.workers = workers
        self.model = None

    def train(self, corpus):
        """corpus: list of strings (code snippets). tokenizes and trains word2vec."""
        tokenized = [tokenize_code(doc) for doc in corpus]
        print(f"training word2vec on {len(tokenized)} documents...")
        self.model = Word2Vec(
            sentences=tokenized,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            workers=self.workers,
            sg=1,  # skip-gram
            epochs=10
        )
        print(f"vocabulary size: {len(self.model.wv)}")
        print(f"embedding dimension: {self.vector_size}")

    def encode(self, texts):
        """mean-pool token vectors to get one vector per text"""
        embeddings = []
        for text in texts:
            tokens = tokenize_code(text)
            vecs = [self.model.wv[t] for t in tokens if t in self.model.wv]
            if vecs:
                embeddings.append(np.mean(vecs, axis=0))
            else:
                embeddings.append(np.zeros(self.vector_size))
        return np.array(embeddings, dtype=np.float32)

## 3. Load CodeXGLUE Dataset

Our knowledge base is built from the [CodeXGLUE](https://github.com/microsoft/CodeXGLUE) Code-to-Text dataset, originally designed for **code summarization** — each example pairs a code method with a natural language description.

We **invert the task** for code generation: the natural language summary + signature becomes the input, and the code becomes the output. This is the same task inversion approach used in prior work on LLMs for code-related tasks.

Each example after inversion:
- **Input:** natural language summary + function signature
- **Output:** the corresponding code implementation

We load both Java and Python training splits to build our knowledge base.

In [5]:
print("loading java dataset...")
ds_java = DatasetDict.load_from_disk(JAVA_DATASET_PATH)
print(f"java train: {len(ds_java['train'])} examples")

print("loading python dataset...")
ds_python = DatasetDict.load_from_disk(PYTHON_DATASET_PATH)
print(f"python train: {len(ds_python['train'])} examples")

# quick look at the format
example = ds_java['train'][0]
print(f"\ninput:\n{example['input'][:300]}...")
print(f"\noutput:\n{example['output'][:200]}...")

loading java dataset...
java train: 164923 examples
loading python dataset...
python train: 251818 examples

input:
Summary: Compare the supplied plaintext password to a hashed password.

@param   passwd  Plaintext password.
@param   hashed  scrypt hashed password.

@return true if passwd matches hashed value.
Signature: public static boolean check(String passwd, String hashed)...

output:
public static boolean check(String passwd, String hashed) {
        try {
            String[] parts = hashed.split("\\$");

            if (parts.length != 5 || !parts[1].equals("s0")) {
            ...


## 4. Process Data for Knowledge Base

Each CodeXGLUE example is parsed into three fields: **summary** (what the code does), **signature** (the function definition), and **code** (the implementation).

For retrieval, we embed `summary + signature` — this is what a user query looks like at inference time ("do X with signature Y"), so embedding the same format ensures the search space matches the query space.

In [6]:
def process_codexglue_example(example, language):
    """extract summary, signature, and code from CodeXGLUE format"""
    input_text = example['input']
    output_code = example['output']

    try:
        summary = input_text.split("Summary:")[1].split("Signature:")[0].strip()
        signature = input_text.split("Signature:")[1].strip()
    except:
        summary = input_text
        signature = ""

    return {
        'summary': summary,
        'signature': signature,
        'code': output_code,
        'language': language,
        'embed_text': f"{summary}\n{signature}"
    }

print("processing java...")
java_data = [process_codexglue_example(ex, 'java') for ex in tqdm(ds_java['train'])]
print(f"java: {len(java_data)} examples")

print("processing python...")
python_data = [process_codexglue_example(ex, 'python') for ex in tqdm(ds_python['train'])]
print(f"python: {len(python_data)} examples")

all_data = java_data + python_data
print(f"\ntotal KB: {len(all_data)} (java: {len(java_data)}, python: {len(python_data)})")

# check one example
print(f"\nlanguage: {all_data[0]['language']}")
print(f"summary: {all_data[0]['summary'][:100]}...")
print(f"signature: {all_data[0]['signature'][:100]}...")
print(f"code: {all_data[0]['code'][:100]}...")

processing java...


100%|██████████| 164923/164923 [00:03<00:00, 43287.88it/s]


java: 164923 examples
processing python...


100%|██████████| 251818/251818 [00:05<00:00, 42980.95it/s]

python: 251818 examples

total KB: 416741 (java: 164923, python: 251818)

language: java
summary: Compare the supplied plaintext password to a hashed password.

@param   passwd  Plaintext password.
...
signature: public static boolean check(String passwd, String hashed)...
code: public static boolean check(String passwd, String hashed) {
        try {
            String[] parts...


## 5. Train Word2Vec and Generate Embeddings

We train Word2Vec on **both** the code implementations and the natural language descriptions. This way the vocabulary covers code tokens (`def`, `return`, `int[]`) and description tokens (`sort`, `convert`, `parse`) — both sides need to live in the same embedding space for retrieval to work.

After training, we encode each KB entry's `summary + signature` into a 128-dim vector. These vectors go into FAISS next.

In [7]:
code_corpus = [item['code'] for item in all_data]
embed_texts = [item['embed_text'] for item in all_data]

embedder = CodeEmbedder(vector_size=128, window=5, min_count=2)
embedder.train(code_corpus + embed_texts)

# encode separately per language
java_embeddings = embedder.encode([item['embed_text'] for item in java_data])
python_embeddings = embedder.encode([item['embed_text'] for item in python_data])

print(f"java embeddings: {java_embeddings.shape}")
print(f"python embeddings: {python_embeddings.shape}")

training word2vec on 833482 documents...
vocabulary size: 1702053
embedding dimension: 128
java embeddings: (164923, 128)
python embeddings: (251818, 128)


## 6. Build FAISS Index (Word2Vec)

We add all embedding vectors to a FAISS `IndexFlatL2` index. At query time, FAISS computes the **L2 (Euclidean) distance** between the query vector and every vector in the index, returning the closest matches.

**L2 distance in simple terms:** Think of each code snippet as a point in space. L2 distance is just the straight-line distance between two points — like measuring with a ruler.

- Distance = **0** means identical
- **Smaller** distance = more similar
- **Larger** distance = less similar
- No upper limit — it depends on how spread out the vectors are
```
Example (2D for simplicity):

  "sort a list"      → point at (3, 4)
  "order an array"   → point at (4, 5)    ← nearby
  "parse JSON"       → point at (9, 1)    ← far away

  L2("sort", "order") = √((4-3)² + (5-4)²) = √2 ≈ 1.4   → small distance, similar task
  L2("sort", "parse") = √((9-3)² + (1-4)²) = √45 ≈ 6.7   → large distance, different task
```

In our case, each point has 128 dimensions instead of 2, but the idea is the same — FAISS finds the nearest points to your query. With ~400K vectors, brute-force exact search runs in milliseconds.

In [10]:
dimension = java_embeddings.shape[1]

java_index = faiss.IndexFlatL2(dimension)
java_index.add(java_embeddings)

python_index = faiss.IndexFlatL2(dimension)
python_index.add(python_embeddings)

print(f"java index: {java_index.ntotal} vectors")
print(f"python index: {python_index.ntotal} vectors")

java index: 164923 vectors
python index: 251818 vectors


## 7. Save Knowledge Base (Word2Vec)

In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

faiss.write_index(java_index, f"{OUTPUT_DIR}/java_index.faiss")
faiss.write_index(python_index, f"{OUTPUT_DIR}/python_index.faiss")

with open(f"{OUTPUT_DIR}/java_metadata.pkl", 'wb') as f:
    pickle.dump(java_data, f)
with open(f"{OUTPUT_DIR}/python_metadata.pkl", 'wb') as f:
    pickle.dump(python_data, f)

embedder.model.save(f"{OUTPUT_DIR}/word2vec.model")

config = {
    'embedding_method': 'word2vec',
    'vector_size': embedder.vector_size,
    'dimension': dimension,
    'java_examples': len(java_data),
    'python_examples': len(python_data)
}
with open(f"{OUTPUT_DIR}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"saved to {OUTPUT_DIR}")

saved to /scratch/mhaque/RAG-Tutorials/dataset/rag_codegen_kb


## 8. Retriever (Word2Vec)

This is the core of the RAG pipeline. Given a query, the retriever:

1. **Embeds** the query using the same Word2Vec model
2. **Selects** the correct FAISS index based on language (Java query → Java index)
3. **Searches** for the nearest vectors
4. **Returns** top-k examples with similarity scores

FAISS returns **L2 distance** (0 = identical, larger = less similar). We convert it to a **similarity score** using `1 / (1 + distance)` so it's easier to interpret:
```
distance = 0    →  1 / (1 + 0)   = 1.0   → identical
distance = 1.4  →  1 / (1 + 1.4) = 0.42  → somewhat similar
distance = 6.7  →  1 / (1 + 6.7) = 0.13  → not similar
```

In [12]:
class CodeRAGRetriever:
    """retriever with separate indexes per language"""

    def __init__(self, kb_dir=OUTPUT_DIR):
        print(f"loading KB from: {kb_dir}")

        with open(f"{kb_dir}/config.json", 'r') as f:
            self.config = json.load(f)

        self.indexes = {
            'java': faiss.read_index(f"{kb_dir}/java_index.faiss"),
            'python': faiss.read_index(f"{kb_dir}/python_index.faiss")
        }

        with open(f"{kb_dir}/java_metadata.pkl", 'rb') as f:
            self.metadata_java = pickle.load(f)
        with open(f"{kb_dir}/python_metadata.pkl", 'rb') as f:
            self.metadata_python = pickle.load(f)

        self.metadata = {'java': self.metadata_java, 'python': self.metadata_python}

        self.embedder = CodeEmbedder(vector_size=self.config['vector_size'])
        self.embedder.model = Word2Vec.load(f"{kb_dir}/word2vec.model")

        print(f"java: {self.config['java_examples']} examples")
        print(f"python: {self.config['python_examples']} examples")

    def retrieve(self, query, language, top_k=3):
        query_embedding = self.embedder.encode([query])

        distances, indices = self.indexes[language].search(query_embedding, top_k)
        meta = self.metadata[language]

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx >= len(meta):
                continue
            item = meta[idx]
            results.append({
                'summary': item['summary'],
                'signature': item['signature'],
                'code': item['code'],
                'language': item['language'],
                'similarity': 1 / (1 + dist)
            })

        return results

w2v_retriever = CodeRAGRetriever()

loading KB from: /scratch/mhaque/RAG-Tutorials/dataset/rag_codegen_kb
java: 164923 examples
python: 251818 examples


## 9. Test Retrieval (Word2Vec)

In [13]:
java_query = "Sort an array of integers in ascending order\npublic void sortArray(int[] arr)"
print(f"query: {java_query}\n")

java_results = w2v_retriever.retrieve(java_query, language='java', top_k=3)

print(f"retrieved {len(java_results)} java examples:")
for i, r in enumerate(java_results, 1):
    print(f"\n  {i}. similarity: {r['similarity']:.4f}")
    print(f"     summary: {r['summary'][:80]}...")
    print(f"     signature: {r['signature'][:60]}...")

query: Sort an array of integers in ascending order
public void sortArray(int[] arr)

retrieved 3 java examples:

  1. similarity: 0.4000
     summary: Sort an array of doubles in descending order.

@param a Values
@return Values in...
     signature: private static double[] reversed(double[] a)...

  2. similarity: 0.3981
     summary: Sort the long array in descending order using this algorithm.

@param longArray ...
     signature: public static void sortDescending(long[] longArray)...

  3. similarity: 0.3836
     summary: Sort the short array in descending order using this algorithm.

@param shortArra...
     signature: public static void sortDescending(short[] shortArray)...


In [14]:
python_query = "Parse a JSON string and return a dictionary\ndef parse_json(json_str: str) -> dict"
print(f"query: {python_query}\n")

python_results = w2v_retriever.retrieve(python_query, language='python', top_k=3)

print(f"retrieved {len(python_results)} python examples:")
for i, r in enumerate(python_results, 1):
    print(f"\n  {i}. similarity: {r['similarity']:.4f}")
    print(f"     summary: {r['summary'][:80]}...")
    print(f"     signature: {r['signature'][:60]}...")

query: Parse a JSON string and return a dictionary
def parse_json(json_str: str) -> dict

retrieved 3 python examples:

  1. similarity: 0.3686
     summary: Given a dictionary from sentence -> extractions,
    return a corresponding CoNL...
     signature: def convert_sent_dict_to_conll(sent_dic, domain) -> str...

  2. similarity: 0.3662
     summary: Convert a JsonObj into a straight dictionary

        :return: dictionary that c...
     signature: def _as_dict(self) -> Dict[str, JsonTypes]...

  3. similarity: 0.3406
     summary: Perform a GET request for the url and return a dictionary parsed from
        th...
     signature: def get_url(url: str) -> Union[dict, int, float, str]...


## 10. Build RAG Context (the "Augmentation" step)

In standard prompting, the model only sees your query. In RAG, we **augment** (enrich) the prompt by adding retrieved examples before the query. The model now has real working code to reference — instead of generating from scratch, it can follow patterns from similar examples.
```
Standard prompt:
  "Sort an array of integers"
  → model guesses from memory

RAG-augmented prompt:
  "Here are similar examples:
     Example 1: [retrieved code]
     Example 2: [retrieved code]
     Example 3: [retrieved code]
   Now generate code for: Sort an array of integers"
  → model follows patterns from real examples
```

This is why it's called Retrieval-**Augmented** Generation — we augment the input with retrieved context before generation.

In [41]:
def build_rag_context(retrieved_examples, language):
    """build few-shot context string from retrieved examples"""
    if not retrieved_examples:
        return ""

    lang_name = "Java" if language == 'java' else "Python"

    parts = [f"Here are similar {lang_name} code examples for reference:\n"]
    for i, ex in enumerate(retrieved_examples, 1):
        parts.append(f"--- Example {i} ---")
        parts.append(f"Task: {ex['summary']}")
        parts.append(f"Signature: {ex['signature']}")
        parts.append(f"Code:\n```{language}\n{ex['code']}\n```\n")
    parts.append(f"Now generate {lang_name} code for the following task:\n")

    return "\n".join(parts)

# preview the augmented prompt
context = build_rag_context(java_results, 'java')
print(context[:2000])

Here are similar Java code examples for reference:

--- Example 1 ---
Task: Sort the cells in increasing order.
Signature: public void order()
Code:
```java
public void order() {
        Collections.sort(cells, new Comparator<SortedSet<Integer>>() {

            @Override
            public int compare(SortedSet<Integer> cellA, SortedSet<Integer> cellB) {
                return cellA.first().compareTo(cellB.first());
            }

        });
    }
```

--- Example 2 ---
Task: Applies this permutation in-place to the elements of an integer array
Signature: public void permute(final int[] arr)
Code:
```java
public void permute(final int[] arr) {
    checkArgument(arr.length == sources.length);
    final int[] tmp = new int[arr.length];
    for (int i = 0; i < tmp.length; ++i) {
      tmp[i] = arr[sources[i]];
    }
    System.arraycopy(tmp, 0, arr, 0, arr.length);
  }
```

--- Example 3 ---
Task: Sorts the terms in ascending order according to their centroids
Signature: public void sor

## 11. Load Generator Model

The third component of RAG — the **generator**. We use Qwen2.5-Coder, a decoder-only LLM designed for code. Unlike CodeT5 (encoder-decoder), decoder-only models work naturally with prompting — you give them text, they continue it. This is exactly what we need: prepend retrieved examples, append the query, and let the model generate code.

In [15]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"loaded {MODEL_NAME} on {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


loaded Qwen/Qwen2.5-Coder-1.5B-Instruct on cpu


## 12. Prompt Building and Code Generation

Two functions that tie everything together:

**`build_prompt`** — Wraps the input into Qwen's chat template with a system message and user message. When RAG context is available, it gets prepended to the user's query. This is where retrieval meets generation.

**`generate_code`** — Feeds the prompt to Qwen and decodes the output. We only decode tokens **after** the input (the model's own generation), not the prompt itself. `temperature=0.0` means greedy decoding — the model always picks the most likely next token, so output is deterministic.

In [24]:
def build_prompt(task_input, rag_context="", language="python"):
    lang_name = "Java" if language == "java" else "Python"
    system = f"You are an expert {lang_name} developer. Generate complete and efficient {lang_name} code based on the given specifications."
    if rag_context:
        system += " Use the provided examples as reference for coding patterns and style."

    user_content = f"{rag_context}\n{task_input}" if rag_context else task_input

    chat = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


def generate_code(prompt, num_samples=1, max_new_tokens=512, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    input_length = inputs['input_ids'].shape[1]

    generations = []
    for _ in range(num_samples):
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature if temperature > 0 else None,
                do_sample=(temperature > 0.0),
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs[0][input_length:]
        code = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        if "Human:" in code:
            code = code.split("Human:")[0].strip()

        generations.append(code)

    return generations

## 13. CoderEval Evaluation Setup

We evaluate our RAG pipeline on [CoderEval](https://github.com/CoderEval/CoderEval) — a benchmark of real-world code generation tasks with executable test cases. Unlike the quick demo above, this runs generation on every task and outputs JSONL files that can be evaluated using CoderEval's Docker environment to compute **pass@k** (the probability that at least one of k generated solutions passes all test cases).

We generate three sets of results — **baseline** (no RAG), **RAG with Word2Vec**, and **RAG with Code2Vec** — to compare whether retrieval helps and which embedding works better.

In [26]:
CODEREVAL_INPUT = "/scratch/mhaque/CoderEval/CoderEval-Input4Models/CEPythonHumanLabel.jsonl"
LANGUAGE = "python"
OUTPUT_DIR_EVAL = "/home/mhaque/RAG-Tutorials/dataset/rag_results"
TOP_K = 3

os.makedirs(OUTPUT_DIR_EVAL, exist_ok=True)

tasks = []
with open(CODEREVAL_INPUT, 'r') as f:
    for line in f:
        if line.strip():
            tasks.append(json.loads(line))

print(f"loaded {len(tasks)} {LANGUAGE} tasks")

loaded 230 python tasks


In [42]:
# some python tasks in CoderEval have unreliable test cases — remove them before generation
if LANGUAGE == "python":
    with open('/home/mhaque/RAG-Tutorials/notebook/ids_to_discard.json', 'r') as f:
        ids_data = json.load(f)
    discard_ids = set(ids_data["CoderEval Python Ids with Unreliable Tests"])

    filtered_tasks = [t for t in tasks if t['question_id'] not in discard_ids]
    print(f"filtered: {len(tasks)} -> {len(filtered_tasks)} tasks ({len(tasks) - len(filtered_tasks)} removed)")
else:
    filtered_tasks = tasks

filtered: 230 -> 190 tasks (40 removed)


In [43]:
# Qwen wraps generated code in markdown blocks (```python ... ```) — strip them for evaluation
def extract_code(text):
    """extract code from markdown blocks"""
    pattern = r'```(?:python|java)?\s*\n?(.*?)```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[0].strip()

    if text.startswith('```python'):
        return text.replace('```python', '', 1).strip()
    if text.startswith('```java'):
        return text.replace('```java', '', 1).strip()
    if text.startswith('```'):
        return text.replace('```', '', 1).strip()

    return text.strip()

## 14. Baseline Inference (No RAG)

In [ ]:
w2v_rag_results = []

for task in tqdm(filtered_tasks, desc="RAG (Word2Vec)"):
    retrieved = w2v_retriever.retrieve(task['input'], language=LANGUAGE, top_k=TOP_K)
    rag_context = build_rag_context(retrieved, LANGUAGE)
    prompt = build_prompt(task['input'], rag_context=rag_context, language=LANGUAGE)
    generations = generate_code(prompt)
    cleaned = [extract_code(g) for g in generations]

    w2v_rag_results.append({
        "_id": task["question_id"],
        "generate_results": cleaned
    })

    if len(w2v_rag_results) % 10 == 0:
        torch.cuda.empty_cache()

w2v_rag_file = f"{OUTPUT_DIR_EVAL}/{LANGUAGE}_rag_w2v_k{TOP_K}.jsonl"
with open(w2v_rag_file, 'w') as f:
    for r in w2v_rag_results:
        f.write(json.dumps(r) + '\n')

print(f"saved {len(w2v_rag_results)} results to {w2v_rag_file}")

baseline:  92%|█████████▏| 175/190 [1:01:20<05:43, 22.90s/it]

## 15. RAG Inference (Word2Vec)

In [ ]:
rag_results = []

for task in tqdm(filtered_tasks, desc="RAG"):
    retrieved = retriever.retrieve(task['input'], language=LANGUAGE, top_k=TOP_K)
    rag_context = build_rag_context(retrieved, LANGUAGE)
    prompt = build_prompt(task['input'], rag_context=rag_context, language=LANGUAGE)
    generations = generate_code(prompt)
    cleaned = [extract_code(g) for g in generations]

    rag_results.append({
        "_id": task["question_id"],
        "generate_results": cleaned
    })

    if len(rag_results) % 10 == 0:
        torch.cuda.empty_cache()

rag_file = f"{OUTPUT_DIR_EVAL}/{LANGUAGE}_rag_k{TOP_K}.jsonl"
with open(rag_file, 'w') as f:
    for r in rag_results:
        f.write(json.dumps(r) + '\n')

print(f"saved {len(rag_results)} results to {rag_file}")

## 16. Code2Vec: Structure-Aware Code Embeddings

Word2Vec treats code as flat text — `public void sort(int[] arr)` is just a sequence of words. But code has **structure**: functions have parameters, loops have bodies, conditions have branches. Two methods can do the same thing using completely different variable names — Word2Vec misses this, because it only sees token co-occurrence.

**Code2Vec** (Alon et al., 2019) learns embeddings from the code's **Abstract Syntax Tree (AST)** — the tree structure that a compiler builds to understand your program.
```
Raw code:     public void sort(int[] arr) { for(int i=0; ...) }

AST:                method_declaration
                   /        |          \
            identifier   parameters     block
              "sort"        |          /    \
                       formal_param   for    ...
                       /      \       
                    type    identifier
                   "int[]"    "arr"
```

Instead of learning from flat tokens, we extract the **sequence of AST node types** (method_declaration, identifier, parameters, ...) and train skip-gram on these structural sequences. Two methods with similar structure get similar embeddings — even if they use completely different variable names.

In [16]:
try:
    from tree_sitter import Language, Parser
    import tree_sitter_java as tsjava
    import tree_sitter_python as tspython
except ImportError:
    os.system("pip install tree-sitter==0.22.3 tree-sitter-java==0.21.0 tree-sitter-python==0.21.0")
    from tree_sitter import Language, Parser
    import tree_sitter_java as tsjava
    import tree_sitter_python as tspython

JAVA_LANG = Language(tsjava.language())
PYTHON_LANG = Language(tspython.language())

java_parser = Parser()
java_parser.language = JAVA_LANG
python_parser = Parser()
python_parser.language = PYTHON_LANG

def extract_ast_nodes(code, language):
    """parse code into AST, return sequence of node types"""
    parser = java_parser if language == 'java' else python_parser
    tree = parser.parse(bytes(code, 'utf-8'))

    nodes = []
    def dfs(node):
        nodes.append(node.type)
        for child in node.children:
            dfs(child)
    dfs(tree.root_node)
    return nodes


class Code2VecEmbedder:
    """structure-aware embeddings using AST node sequences + skip-gram"""

    def __init__(self, vector_size=128, window=5, min_count=2):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.model = None

    def train(self, data):
        """data: list of dicts with 'code' and 'language' keys"""
        print("parsing ASTs...")
        ast_sequences = []
        for item in tqdm(data):
            try:
                nodes = extract_ast_nodes(item['code'], item['language'])
                ast_sequences.append(nodes)
            except:
                ast_sequences.append([])

        print(f"training skip-gram on {len(ast_sequences)} AST sequences...")
        self.model = Word2Vec(
            sentences=ast_sequences,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=1,
            epochs=10
        )
        print(f"AST vocabulary size: {len(self.model.wv)}")

    def encode(self, data):
        """mean-pool AST node vectors to get one vector per code snippet"""
        embeddings = []
        for item in data:
            try:
                nodes = extract_ast_nodes(item['code'], item['language'])
                vecs = [self.model.wv[n] for n in nodes if n in self.model.wv]
                if vecs:
                    embeddings.append(np.mean(vecs, axis=0))
                else:
                    embeddings.append(np.zeros(self.vector_size))
            except:
                embeddings.append(np.zeros(self.vector_size))
        return np.array(embeddings, dtype=np.float32)

In [ ]:
c2v_embedder = Code2VecEmbedder(vector_size=128, window=5, min_count=2)
c2v_embedder.train(all_data)

java_embeddings_c2v = c2v_embedder.encode(java_data)
python_embeddings_c2v = c2v_embedder.encode(python_data)

print(f"code2vec — java: {java_embeddings_c2v.shape}, python: {python_embeddings_c2v.shape}")

parsing ASTs...


100%|██████████| 416741/416741 [00:59<00:00, 7015.29it/s] 


training skip-gram on 416741 AST sequences...


## 17. Build FAISS Index (Code2Vec)

Same FAISS setup as Word2Vec, but with Code2Vec embeddings.

In [ ]:
c2v_java_index = faiss.IndexFlatL2(dimension)
c2v_java_index.add(java_embeddings_c2v)

c2v_python_index = faiss.IndexFlatL2(dimension)
c2v_python_index.add(python_embeddings_c2v)

print(f"code2vec indexes — java: {c2v_java_index.ntotal}, python: {c2v_python_index.ntotal}")

In [ ]:
C2V_OUTPUT_DIR = f"{OUTPUT_DIR}_c2v"
os.makedirs(C2V_OUTPUT_DIR, exist_ok=True)

faiss.write_index(c2v_java_index, f"{C2V_OUTPUT_DIR}/java_index.faiss")
faiss.write_index(c2v_python_index, f"{C2V_OUTPUT_DIR}/python_index.faiss")

with open(f"{C2V_OUTPUT_DIR}/java_metadata.pkl", 'wb') as f:
    pickle.dump(java_data, f)
with open(f"{C2V_OUTPUT_DIR}/python_metadata.pkl", 'wb') as f:
    pickle.dump(python_data, f)

c2v_embedder.model.save(f"{C2V_OUTPUT_DIR}/word2vec.model")

config = {
    'embedding_method': 'code2vec',
    'vector_size': c2v_embedder.vector_size,
    'dimension': dimension,
    'java_examples': len(java_data),
    'python_examples': len(python_data)
}
with open(f"{C2V_OUTPUT_DIR}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"saved to {C2V_OUTPUT_DIR}")

## 18. Retriever (Code2Vec)

In [ ]:
class Code2VecRAGRetriever:
    """retriever using code2vec embeddings"""

    def __init__(self, kb_dir, c2v_embedder):
        print(f"loading KB from: {kb_dir}")

        with open(f"{kb_dir}/config.json", 'r') as f:
            self.config = json.load(f)

        self.indexes = {
            'java': faiss.read_index(f"{kb_dir}/java_index.faiss"),
            'python': faiss.read_index(f"{kb_dir}/python_index.faiss")
        }

        with open(f"{kb_dir}/java_metadata.pkl", 'rb') as f:
            self.metadata_java = pickle.load(f)
        with open(f"{kb_dir}/python_metadata.pkl", 'rb') as f:
            self.metadata_python = pickle.load(f)

        self.metadata = {'java': self.metadata_java, 'python': self.metadata_python}
        self.embedder = c2v_embedder

        print(f"java: {self.config['java_examples']} examples")
        print(f"python: {self.config['python_examples']} examples")

    def retrieve(self, query, language, top_k=3):
        # encode query as AST
        query_data = [{'code': query, 'language': language}]
        query_embedding = self.embedder.encode(query_data)

        distances, indices = self.indexes[language].search(query_embedding, top_k)
        meta = self.metadata[language]

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx >= len(meta):
                continue
            item = meta[idx]
            results.append({
                'summary': item['summary'],
                'signature': item['signature'],
                'code': item['code'],
                'language': item['language'],
                'similarity': 1 / (1 + dist)
            })

        return results

c2v_retriever = Code2VecRAGRetriever(C2V_OUTPUT_DIR, c2v_embedder)

In [ ]:
java_results_c2v = c2v_retriever.retrieve(java_query, language='java', top_k=3)

print(f"retrieved {len(java_results_c2v)} java examples:")
for i, r in enumerate(java_results_c2v, 1):
    print(f"\n  {i}. similarity: {r['similarity']:.4f}")
    print(f"     summary: {r['summary'][:80]}...")
    print(f"     signature: {r['signature'][:60]}...")

## 19. RAG Inference (Code2Vec)

In [ ]:
c2v_rag_results = []

for task in tqdm(filtered_tasks, desc="RAG (Code2Vec)"):
    retrieved = c2v_retriever.retrieve(task['input'], language=LANGUAGE, top_k=TOP_K)
    rag_context = build_rag_context(retrieved, LANGUAGE)
    prompt = build_prompt(task['input'], rag_context=rag_context, language=LANGUAGE)
    generations = generate_code(prompt)
    cleaned = [extract_code(g) for g in generations]

    c2v_rag_results.append({
        "_id": task["question_id"],
        "generate_results": cleaned
    })

    if len(c2v_rag_results) % 10 == 0:
        torch.cuda.empty_cache()

c2v_rag_file = f"{OUTPUT_DIR_EVAL}/{LANGUAGE}_rag_c2v_k{TOP_K}.jsonl"
with open(c2v_rag_file, 'w') as f:
    for r in c2v_rag_results:
        f.write(json.dumps(r) + '\n')

print(f"saved {len(c2v_rag_results)} results to {c2v_rag_file}")

## Side-by-Side: Word2Vec vs Code2Vec Retrieval

Same query, two different embedding models. Compare what each retriever finds.

In [ ]:
test_queries = [
    ("Sort an array of integers in ascending order\npublic void sortArray(int[] arr)", "java"),
    ("Parse a JSON string and return a dictionary\ndef parse_json(json_str: str) -> dict", "python"),
    ("Check if a string is a valid email address\ndef is_valid_email(email: str) -> bool", "python"),
]

for query, lang in test_queries:
    print(f"query: {query[:60]}...")
    print(f"language: {lang}\n")

    w2v_results = w2v_retriever.retrieve(query, language=lang, top_k=3)
    c2v_results = c2v_retriever.retrieve(query, language=lang, top_k=3)

    print("  Word2Vec results:")
    for i, r in enumerate(w2v_results, 1):
        print(f"    {i}. [{r['similarity']:.4f}] {r['summary'][:70]}...")

    print("\n  Code2Vec results:")
    for i, r in enumerate(c2v_results, 1):
        print(f"    {i}. [{r['similarity']:.4f}] {r['summary'][:70]}...")

    print("\n" + "=" * 60 + "\n")

## 20. Summary

Three JSONL files ready for CoderEval Docker evaluation: